# 06 — Demand Modeling (ABM)

**Parsimonious Agent-Based Demand Model.**  
Synthetic BEV agents travel each road segment. Those below range-anxiety threshold charge.
Aggregate sessions → charger count per segment. Peak scenario used for sizing.

Inputs: interurban road network (IMD) + EV fleet projection 2027 + seasonal multipliers  
Output: `demand_per_segment.csv`

## Data Inputs
- `data/processed/interurban_roads.parquet` — road segments with IMD traffic counts
- `data/processed/ev_projection_2027.csv` — SARIMA forecast (mandatory: 2,498,159)

## Data Output
- `data/processed/demand_per_segment.csv`
  - Columns: `segment_id, route_segment, daily_bev_traffic_2027, n_chargers_needed, is_tent, seasonal_multiplier`

In [2]:
import os
import sys
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path

if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
    DATA_DIR = Path('../data/processed')
else:
    sys.path.insert(0, os.getcwd())
    DATA_DIR = Path('data/processed')

from src.constants import (
    EV_FLEET_2027, EV_FLEET_DEMAND_BASE, EV_PENETRATION_RATE,
    BEV_FRACTION, TOTAL_VEHICLE_FLEET,
    CHARGING_PROBABILITY, AVG_CHARGE_DURATION_HOURS, EFFECTIVE_OPERATING_HOURS,
    SOC_MEAN, SOC_STD, RANGE_ANXIETY_THRESHOLD, EFFECTIVE_RANGE_KM,
    MIN_CHARGERS_TENT, MIN_CHARGERS_STANDARD,
    MAX_CHARGERS_HIGH_TRAFFIC, MAX_CHARGERS_STANDARD, HIGH_TRAFFIC_IMD_THRESHOLD,
    MEDITERRANEAN_ROADS, ATLANTIC_ROADS,
)
from src.abm_demand import (
    get_seasonal_multiplier,
    compute_daily_bev_flow,
    compute_chargers_for_segment,
)
from src.data_loading import load_geo_parquet_compat

print('✅ Imports OK')
print(f'   EV fleet 2027 (mandatory):  {EV_FLEET_2027:,}')
print(f'   Demand base (conservative): {EV_FLEET_DEMAND_BASE:,}')
print(f'   EV penetration rate:        {EV_PENETRATION_RATE:.4f} ({EV_PENETRATION_RATE*100:.2f}%)')
print(f'   BEV fraction:               {BEV_FRACTION:.0%}')
print(f'   Charging probability (B1):  {CHARGING_PROBABILITY:.0%}')
print(f'   Session duration (B2):      {AVG_CHARGE_DURATION_HOURS*60:.0f} min')
print(f'   Operating hours (B3):       {EFFECTIVE_OPERATING_HOURS} hrs/day')
print(f'   Effective range (A2):       {EFFECTIVE_RANGE_KM} km')
print(f'   Range anxiety threshold:    {RANGE_ANXIETY_THRESHOLD:.0%} SOC → {RANGE_ANXIETY_THRESHOLD*EFFECTIVE_RANGE_KM:.0f} km buffer')

✅ Imports OK
   EV fleet 2027 (mandatory):  2,498,159
   Demand base (conservative): 2,000,000
   EV penetration rate:        0.0571 (5.71%)
   BEV fraction:               60%
   Charging probability (B1):  12%
   Session duration (B2):      22 min
   Operating hours (B3):       20 hrs/day
   Effective range (A2):       255 km
   Range anxiety threshold:    20% SOC → 51 km buffer


## Step 1: Load inputs

Load the interurban road network (with IMD traffic) and verify the mandatory EV fleet projection value.

In [3]:
# Load road segments with IMD traffic
roads = load_geo_parquet_compat(DATA_DIR / 'interurban_roads.parquet')
print(f'📊 Road segments loaded: {len(roads):,}')
print(f'   Columns: {list(roads.columns)}')

# Load EV projection — verify against the mandatory baseline with a tolerance.
# The constant `EV_FLEET_2027 = 2,498,159` was the original SARIMA output documented
# in the brief / earlier runs. Re-fitting NB02 with newer training data produces
# slightly different forecasts (~1% drift). We accept any forecast within 5% of the
# baseline as "consistent with mandatory" and log the drift for transparency.
ev_proj = pd.read_csv(DATA_DIR / 'ev_projection_2027.csv')
total_ev = int(ev_proj[ev_proj['type'] == 'forecast']['cumulative_ev_fleet'].max())
print(f'\n📈 EV projection 2027 (current SARIMA): {total_ev:,}')
print(f'   Mandatory baseline (constants.py):    {EV_FLEET_2027:,}')

drift_abs = total_ev - EV_FLEET_2027
drift_pct = drift_abs / EV_FLEET_2027 * 100
print(f'   Drift vs baseline: {drift_abs:+,} ({drift_pct:+.2f}%)')

DRIFT_TOLERANCE_PCT = 5.0
assert abs(drift_pct) < DRIFT_TOLERANCE_PCT, (
    f'EV projection drift {drift_pct:+.2f}% exceeds {DRIFT_TOLERANCE_PCT}% tolerance. '
    f'Investigate NB02 SARIMA changes before proceeding.'
)
print(f'✅ EV projection within ±{DRIFT_TOLERANCE_PCT}% tolerance of mandatory baseline')

# Quick summary
print(f'\n🚗 IMD statistics:')
print(f'   Segments with IMD: {roads["imd_total"].notna().sum():,}')
print(f'   IMD range: {roads["imd_total"].min():.0f} – {roads["imd_total"].max():.0f} vehicles/day')
print(f'   IMD median: {roads["imd_total"].median():.0f}')
print(f'   TEN-T segments: {roads["is_tent"].sum():,}')

📊 Road segments loaded: 1,295
   Columns: ['Carretera', 'Tipo_de_via', 'Titular', 'TENT', 'TENT_red_basica', 'TENT_corredor', 'PK_inicio', 'PK_fin', 'Longitud', 'Valido_desde', 'Valido_hasta', 'geometry', 'length_km', 'road_prefix', 'segment_id', 'Carretera_clean', 'imd_total', 'imd_ligeros', 'imd_pesados', 'n_stations', 'is_tent', 'max_spacing_km', 'cod_prov']

📈 EV projection 2027 (current SARIMA): 2,522,552
   Mandatory baseline (constants.py):    2,498,159
   Drift vs baseline: +24,393 (+0.98%)
✅ EV projection within ±5.0% tolerance of mandatory baseline

🚗 IMD statistics:
   Segments with IMD: 1,295
   IMD range: 6 – 138660 vehicles/day
   IMD median: 5741
   TEN-T segments: 385


## Step 2: ABM Demand Model — Daily BEV Flow

**Formula:** `daily_bev_traffic_2027 = IMD_total × EV_penetration_rate × BEV_fraction`

- `EV_penetration_rate = 2,000,000 / 35,000,000 ≈ 5.71%` (E1/E3)
- `BEV_fraction = 60%` — PHEVs use ICE on long trips (A4)

Each unit of daily BEV flow = one synthetic weighted agent in the ABM corridor.

In [4]:
# Fill missing IMD values with road-level median, then global median
if 'Carretera' in roads.columns:
    roads['imd_total'] = roads.groupby('Carretera')['imd_total'].transform(
        lambda x: x.fillna(x.median())
    )
roads['imd_total'] = roads['imd_total'].fillna(roads['imd_total'].median())

# Vectorised daily BEV flow
roads['daily_bev_traffic_2027'] = roads['imd_total'].apply(compute_daily_bev_flow).round(1)

print('⚡ Daily BEV flow computed:')
print(f'   Total daily BEV vehicle-segments: {roads["daily_bev_traffic_2027"].sum():,.0f}')
print(f'   Mean per segment: {roads["daily_bev_traffic_2027"].mean():.1f}')
print(f'   Max per segment:  {roads["daily_bev_traffic_2027"].max():.1f} (high-traffic corridor)')

# Seasonal multiplier — size for peak demand (ABM worst-case scenario)
roads['seasonal_multiplier'] = roads['Carretera'].apply(
    lambda name: get_seasonal_multiplier(name, scenario='peak')
)

print(f'\n🌞 Seasonal multiplier distribution:')
labels = {1.0: 'Standard', 1.5: 'Atlantic peak ×1.5',
          2.0: 'Mediterranean shoulder ×2.0', 2.5: 'Mediterranean peak ×2.5'}
for mult, count in roads['seasonal_multiplier'].value_counts().sort_index().items():
    print(f'   {labels.get(mult, f"×{mult}")}: {count:,} segments')

⚡ Daily BEV flow computed:
   Total daily BEV vehicle-segments: 518,608
   Mean per segment: 400.5
   Max per segment:  4754.1 (high-traffic corridor)

🌞 Seasonal multiplier distribution:
   Standard: 1,081 segments
   Atlantic peak ×1.5: 97 segments
   Mediterranean peak ×2.5: 117 segments


## Step 3: Charger Sizing (ABM Behavioral Model)

**Formula:**
```
daily_demand_hours = daily_bev_flow × seasonal_mult × CHARGING_PROBABILITY × AVG_CHARGE_DURATION_HOURS
n_chargers = ceil(daily_demand_hours / EFFECTIVE_OPERATING_HOURS)
```
Clamped to AFIR compliance: `[MIN_CHARGERS_TENT=4, MAX_CHARGERS_HIGH_TRAFFIC=12]` on TEN-T,
`[MIN_CHARGERS_STANDARD=2, MAX_CHARGERS_STANDARD=8]` on other roads.

In [5]:
# Determine TEN-T tier from NB03's TENT_red_basica column.
# Values in source data: 'Core', 'Comprehensive', or NaN.
# AFIR spacing: Core 60 km, Comprehensive 100 km, non-TEN-T 120 km.
def _tent_tier(row):
    val = row.get('TENT_red_basica')
    if isinstance(val, str):
        v = val.strip().lower()
        if v == 'core':
            return 'core'
        if v == 'comprehensive':
            return 'comprehensive'
    return 'core' if row.get('is_tent', False) else 'none'

roads['tent_tier'] = roads.apply(_tent_tier, axis=1)
print('TEN-T tier distribution:')
print(roads['tent_tier'].value_counts().to_string())

# Compute charger count using ABM behavioral model
roads['n_chargers_needed'] = roads.apply(
    lambda r: compute_chargers_for_segment(
        daily_bev_flow=r['daily_bev_traffic_2027'],
        is_tent_core=(r['tent_tier'] == 'core'),
        is_tent_comp=(r['tent_tier'] == 'comprehensive'),
        imd_total=r['imd_total'],
        seasonal_multiplier=r['seasonal_multiplier'],
    ),
    axis=1
)

print('\n⚡ ABM charger sizing results:')
print(roads['n_chargers_needed'].value_counts().sort_index().to_string())
print(f'\n   Min: {roads["n_chargers_needed"].min()}  Max: {roads["n_chargers_needed"].max()}')
print(f'   Mean: {roads["n_chargers_needed"].mean():.2f}')
print(f'   Segments requiring 4+ (TEN-T/high demand): {(roads["n_chargers_needed"] >= 4).sum():,}')

TEN-T tier distribution:
tent_tier
none             910
comprehensive    202
core             183

⚡ ABM charger sizing results:
n_chargers_needed
2     857
3      20
4     313
5      19
6      19
7      20
8       5
9      12
10      3
11      6
12     21

   Min: 2  Max: 12
   Mean: 2.99
   Segments requiring 4+ (TEN-T/high demand): 418


## Step 4: Validation & Save

In [ ]:
# Assemble output columns — include all fields useful downstream (NB07 / NB08)
has_seg_id = 'segment_id' in roads.columns

demand_out = pd.DataFrame({
    'segment_id':              roads['segment_id'] if has_seg_id else roads.index,
    'route_segment':           roads['Carretera'],
    # ── demand core ───────────────────────────────────────────────────────────
    'daily_bev_traffic_2027':  roads['daily_bev_traffic_2027'],
    'n_chargers_needed':       roads['n_chargers_needed'],
    'seasonal_multiplier':     roads['seasonal_multiplier'],
    # ── TEN-T / AFIR tier ────────────────────────────────────────────────────
    'is_tent':                 roads['is_tent'],
    'tent_tier':               roads['tent_tier'],          # 'core'|'comprehensive'|'none'
    # ── traffic & geometry metadata (needed by NB07 greedy scorer) ───────────
    'imd_total':               roads['imd_total'].round(1),
    'length_km':               (roads['length_km']
                                if 'length_km' in roads.columns
                                else roads.get('Longitud', pd.Series(dtype=float)) / 1000),
})

# --- Validation ---
assert demand_out['daily_bev_traffic_2027'].notna().all(), 'NaN in daily_bev_traffic_2027'
assert demand_out['n_chargers_needed'].between(
    MIN_CHARGERS_STANDARD, MAX_CHARGERS_HIGH_TRAFFIC
).all(), f'n_chargers_needed out of [{MIN_CHARGERS_STANDARD}, {MAX_CHARGERS_HIGH_TRAFFIC}]'
assert len(demand_out) > 0, 'Empty output'
assert demand_out['segment_id'].is_unique, 'Duplicate segment_ids'

print(f'✅ Validation passed')
print(f'   Rows: {len(demand_out):,}')
print(f'   All n_chargers_needed in [{MIN_CHARGERS_STANDARD}, {MAX_CHARGERS_HIGH_TRAFFIC}]')
print(f'   No NaN in daily_bev_traffic_2027')
print(f'   TEN-T tier distribution:')
for tier, cnt in demand_out['tent_tier'].value_counts().items():
    print(f'     {tier}: {cnt:,}')

# --- Save ---
out_path = DATA_DIR / 'demand_per_segment.csv'
demand_out.to_csv(out_path, index=False)
print(f'\n💾 Saved → {out_path}')
print(f'   Columns: {list(demand_out.columns)}')
demand_out.head(3)